 # Taller polars y parquet avanzado

**John Gonzalez**

*19 de octubre de 2026*

---

## Objetivo

Imaginen que ustedes trabajan en una empresa que procesan datos bigdata, un ing junior genera código con IA, pero está lleno de problemas. Tu objetivo es corregirlos.

## Entorno de Trabajo

* **Dataset:** Predicción de tarifas de Taxi de Nueva York (5.4 GB en formato CSV, descárgala de kaggle siguiendo las instrucciones vistas en clase, use como referencia el notebook 05).

* **Entorno:** GitHub Codespaces (Límite de recursos: 8 GB RAM).

## A tener en cuenta

* Cuando se le pida "consulte ..." puede apoyarse de la IA para investigar y entender, muy recomendable.
* Cuando se le pida "escriba código" puede apoyarse de la IA, pero debe entender 100% el código generado, se validará en clase el entendimiento, posiblmente se solicite el prompt utilizado para llegar a la solución.
* Cuando se le pida "escriba con sus propias palabras ..." puede parafrasear pero lo escrito debe ser 100% escrito por ud mismo.
* Este taller es la continuación del 07, se recomienda ejecutarlo y entenderlo antes de iniciar.
* Este taller aborda nuevos conceptos, es intencional.

In [2]:
# Configuración del Entorno e Importación de Librerías
import polars as pl
import os
import time

# Rutas de archivos de trabajo
path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"
archivo_csv = path + "/train.csv"
archivo_parquet = path + "/train.parquet"

print("Versión de Polars instalada:", pl.__version__)


Versión de Polars instalada: 2.0.0


 ---

 ## Caso 1: Evaluación Ansiosa vs. Perezosa (Gestión de Memoria RAM)



 **Escenario:** Un analista intenta cargar el archivo completo `train.csv` (5.4 GB) a la memoria RAM utilizando evaluación ansiosa (*Eager Execution*) para calcular la tarifa promedio (`fare_amount`). El sistema colapsa debido a un desbordamiento de memoria (*Out of Memory - OOM*).

 > **Tip:** En clase vimos el método `scan_csv`. Puedes ejecutar el cálculo por lotes en disco (*Out-of-Core*) agregando el argumento `streaming=True` al método `.collect()`.

In [2]:
# CÓDIGO INEFICIENTE (CASO 1)
print("--- Caso 1: Intentando lectura ansiosa (Eager) ---")
try:
    inicio = time.time()
    # Ineficiencia: Carga ansiosa que sobrepasa la memoria física disponible
    df = pl.read_csv(archivo_csv)
    media_tarifa = df.select(pl.col("fare_amount").mean())
    print(f"Resultado: {media_tarifa} | Tiempo de ejecución: {time.time() - inicio:.2f} s")
except Exception as e:
    print(f"Error de ejecución detectado (OOM): {e}")


--- Caso 1: Intentando lectura ansiosa (Eager) ---


: 

 ### [Espacio de Respuesta - Caso 1]

 **Escribe tu código corregido a continuación y documenta:**

 1. El tiempo de ejecución sin streaming con `.collect()`.
 2. El tiempo de ejecución alcanzado con la solución perezosa en modo *streaming* con `.collect(streaming=True)`.
 3. Consulta y explica con tus propias palabras, por qué el procesamiento por lotes (*Out-of-Core*) podría evitar el colapso de la memoria RAM.

In [ ]:
# Escribe tu código aquí para el caso 1

print("--- Caso 1: Evaluación perezosa (Lazy) ---")

# Plan perezoso: no lee datos todavía, solo define la consulta
plan = (
    pl.scan_csv(archivo_csv)
    .select(pl.col("fare_amount").mean().alias("media_fare_amount"))
)

# Opcional: muestra el plan optimizado 
print(plan.explain())

# 1. Ejecución perezosa SIN streaming: .collect()
inicio = time.time()
resultado_normal = plan.collect()
tiempo_normal = time.time() - inicio
print(f"[collect()] Resultado: {resultado_normal.item():.4f} | Tiempo: {tiempo_normal:.2f} s")

# 2. Ejecución perezosa CON streaming: .collect(streaming=True)
inicio = time.time()
resultado_streaming = plan.collect(engine="streaming")
tiempo_streaming = time.time() - inicio
print(f"[collect(engine='streaming')] Resultado: {resultado_streaming.item():.4f} | Tiempo: {tiempo_streaming:.2f} s")

El procesamiento por lotes evita el colapso de RAM porque no carga el archivo completo en memoria. Lee el archivo en trozos pequeños desde el disco, procesa cada trozo y guarda solo un resultado parcial. Luego libera ese trozo y pasa al siguiente. Así, el uso de RAM depende del tamaño del lote y no del tamaño del archivo.

En Polars esto se activa con scan_csv (plan perezoso) y collect(engine="streaming"). Solo se leen las columnas necesarias y se procesa por partes.

In [ ]:
# Escribe tu código aquí para el caso 2

 ---

 ## Caso 2: Inferencia de Tipos y Filtros Temporales (Fechas como String vs. Datetime)



 **Escenario:** Para filtrar las transacciones correspondientes al mes de diciembre de 2015, se aplican expresiones regulares sobre la columna `pickup_datetime` leída como texto (`String`). La consulta requiere escanear millones de cadenas de caracteres, resultando extremadamente lenta.

 > **Tip:** No alcazamos a ver como convertir texto a formato fecha-hora. Pero es clave porque filtrar cadenas de texto en conjuntos de datos masivos inhabilita las optimizaciones vectoriales. Convierte la columna a formato temporal nativo mediante `.str.to_datetime("%Y-%m-%d %H:%M:%S UTC")` o `.cast(pl.Datetime)`. Una vez casteada, utiliza expresiones temporales nativas como `.dt.year() == 2015` y `.dt.month() == 12`.

In [4]:
# CODIGO INEFICIENTE (CASO 2)
print("--- Caso 2: Búsqueda sobre cadenas de texto ---")
inicio = time.time()

# Ineficiencia: Coincidencia de cadenas de texto sobre 55 millones de registros
resultado_cadenas = (
    pl.scan_csv(archivo_csv)
    .filter(pl.col("pickup_datetime").str.contains("2015-12"))
    .select(pl.len())
    .collect(engine="streaming")
)

tiempo_cadenas = time.time() - inicio
print(f"Registros encontrados en Dic 2015: {resultado_cadenas.item(0,0)} | Tiempo: {tiempo_cadenas:.2f} s")

--- Caso 2: Búsqueda sobre cadenas de texto ---
Registros encontrados en Dic 2015: 0 | Tiempo: 448.62 s


 ### [Espacio de Respuesta - Caso 2]

 **Escribe tu código corregido a continuación y documenta:**

 1. El tiempo de ejecución tras convertir la columna a `Datetime`.

 2. La ganancia porcentual en velocidad frente al escaneo de cadenas de texto.

In [5]:
# Escribe tu código aquí
print("--- Caso 2: Filtro con fechas nativas (Datetime) ---")
inicio = time.time()

resultado_fechas = (
    # 1. Lectura perezosa: solo define el plan, no carga datos en RAM
    pl.scan_csv(archivo_csv)

    # 2. Convertir texto a fecha-hora nativa (Datetime).
    #    El formato debe coincidir con el CSV, por ejemplo: "2015-12-05 14:30:00 UTC".
    #    strict=False: filas con formato raro quedan en null en vez de detener el proceso.
    .with_columns(
        pl.col("pickup_datetime")
        .str.to_datetime("%Y-%m-%d %H:%M:%S UTC", strict=False)
    )

    # 3. Filtro con expresiones temporales nativas (vectorizadas):
    #    comparan números (año y mes) en lugar de buscar texto en cada fila.
    .filter(
        (pl.col("pickup_datetime").dt.year() == 2015)
        & (pl.col("pickup_datetime").dt.month() == 12)
    )

    # 4. Solo necesitamos contar las filas que cumplen el filtro
    .select(pl.len())

    # 5. Ejecución por lotes (out-of-core).
    #    En Polars >= 2.0, streaming=True se reemplaza por engine="streaming".
    .collect(engine="streaming")
)

tiempo_fechas = time.time() - inicio

# DOCUMENTACIÓN
# Instrucción 1: tiempo de ejecución tras convertir la columna a Datetime
print(f"Registros encontrados en Dic 2015: {resultado_fechas.item(0,0)}")
print(f"1) Tiempo con Datetime: {tiempo_fechas:.2f} s")

# Instrucción 2: ganancia porcentual en velocidad frente al escaneo de cadenas
# (tiempo_cadenas viene de la celda del código ineficiente)
ganancia = (tiempo_cadenas - tiempo_fechas) / tiempo_cadenas * 100
print(f"   Tiempo con cadenas: {tiempo_cadenas:.2f} s")
print(f"2) Ganancia porcentual: {ganancia:.1f} %")

# Verificación: ambos métodos deben contar lo mismo
print("Conteos coinciden:", resultado_cadenas.item(0,0) == resultado_fechas.item(0,0))

--- Caso 2: Filtro con fechas nativas (Datetime) ---
Registros encontrados en Dic 2015: 0
1) Tiempo con Datetime: 33.07 s
   Tiempo con cadenas: 448.62 s
2) Ganancia porcentual: 92.6 %
Conteos coinciden: True


 ---

 ## Caso 3: Preservación del Plan Lógico de Ejecución (DAG y `.explain()`)



 **Escenario:** En la línea inicial del pipeline, se convierte el `LazyFrame` a `DataFrame` ejecutando `.collect()`. Posteriormente, se aplican los filtros geográficos y la selección de columnas sobre el objeto en memoria.



 > **Recomendación:** Similar a `Spark`, materializar los datos usando `collect` de forma prematura destruye el Grafo Acíclico Dirigido (DAG). Esto impide que el optimizador de Polars aplique *Projection Pushdown* (seleccionar únicamente las columnas requeridas desde el almacenamiento) y *Selection Pushdown* (filtrar filas antes de cargar los datos). Conserva la cadena como `LazyFrame` e inspecciona el plan optimizado invocando `.explain()`.

In [6]:
# CÓDIGO INEFICIENTE (CASO 3)
print("--- Caso 3: Materialización prematura del DataFrame ---")
inicio = time.time()

# Ineficiencia: Invocación prematura de .collect() que anula las optimizaciones del DAG
# NOTA: Es muy posible que esta linea desborde la memoria y tengas que reiniciar el kernel
df_materializado = pl.scan_parquet(archivo_parquet).collect()

# Las operaciones de filtrado y proyección se ejecutan de forma ansiosa sobre todo el conjunto
resultado_suboptimo = (
    df_materializado
    .filter(pl.col("passenger_count") == 1)
    .select("fare_amount")
)

tiempo_suboptimo = time.time() - inicio
print(f"Tiempo total con materialización prematura: {tiempo_suboptimo:.2f} s")


--- Caso 3: Materialización prematura del DataFrame ---


: 

 ### [Espacio de Respuesta - Caso 3]

 **Escribe tu código corregido a continuación y documenta:**

 1. El código reordenado donde `.collect()` se ubica únicamente al final del procesamiento.

 2. La salida del comando `print(lazy_query.explain())`, señalando dónde el motor aplicó *Selection Pushdown* y *Projection Pushdown*.

In [3]:
# Escribe el código del punto 1 aqui
print("--- Caso 3: Plan perezoso preservado (DAG intacto) ---")
inicio = time.time()

# Todo el pipeline se construye como LazyFrame: aquí NO se lee ningún dato.
# scan_parquet solo registra el origen en el plan de ejecución.
lazy_query = (
    pl.scan_parquet(archivo_parquet)
    .filter(pl.col("passenger_count") == 1)   # filtro de filas
    .select("fare_amount")                    # columnas requeridas
)

# .collect() se invoca UNA sola vez, al final, cuando el optimizador ya conoce
# el plan completo y puede empujar el filtro y la selección hasta la lectura.
resultado_optimo = lazy_query.collect()

tiempo_optimo = time.time() - inicio
print(f"Filas devueltas: {resultado_optimo.height}")
print(f"Tiempo con plan perezoso (collect al final): {tiempo_optimo:.2f} s")

--- Caso 3: Plan perezoso preservado (DAG intacto) ---
Filas devueltas: 38337524
Tiempo con plan perezoso (collect al final): 42.81 s


In [4]:
# Escribe el código del punto 2 aqui
print("--- Plan optimizado de la consulta (.explain()) ---")
print(lazy_query.explain())

--- Plan optimizado de la consulta (.explain()) ---
simple π 1/1 ["fare_amount"]
  Parquet SCAN [/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction/train.parquet]
  PROJECT 2/8 COLUMNS
  SELECTION: col("passenger_count") == 1
  ESTIMATED ROWS: 55423856


 ---

 ## Caso 4: Particionamiento Físico y el "Small File Problem"



 **Escenario:** Con el propósito de organizar un Data Lake en disco, se intenta particionar el conjunto de datos en formato Parquet utilizando la columna de coordenada de longitud (`pickup_longitude`).

En clase vimos que los archivos parquet se dividen en bloques. Esto es muy beneficioso porque cada bloque tiene metadatos como el valor míni y máximo, si por ejemplo tenemos un filtro como `ickup_longitude < -74`, polars leerá solo los bloques que cumplen con esta condición. Además, también existe la posibilidad de dividir los datos en pequeños archivos o carpetas parquet, de tal manera que polars solo se enfocará en leer aquellos archivos que cumplen alguna condición por ejemplo `year=2015`. 


 > **Recomendación:** Polars ofrece la oportunidad de particionar por variable, sin embargo, cuando la partición se hace por variables continuas puede generar miles de directorios y archivos Parquet diminutos (de pocos kilobytes). Esto provoca latencias severas en el sistema de archivos (Disk I/O Latency). Selecciona columnas discretas o categóricas de **baja cardinalidad** (ej. `year` y `month`) para mantener tamaños óptimos (100 MB - 500 MB).

In [5]:
# CÓDIGO INEFICIENTE (CASO 4)
print("--- Caso 4: Particionamiento por variable continua (Alta Cardinalidad) ---")
# ADVERTENCIA: La siguiente llamada se limita conceptualmente para evitar la creación masiva de archivos.
try:
    # Ineficiencia: Intento de particionamiento sobre una columna flotante continua
    pl.scan_parquet(archivo_parquet).sink_parquet(
        "parquet_inadecuado",
        partition_by=["pickup_longitude"]
    )
except Exception as e:
    print(f"Respuesta del sistema ante particionamiento inadecuado: {e}")


--- Caso 4: Particionamiento por variable continua (Alta Cardinalidad) ---
Respuesta del sistema ante particionamiento inadecuado: LazyFrame.sink_parquet() got an unexpected keyword argument 'partition_by'


 ### [Espacio de Respuesta - Caso 4]

 **Escribe tu código corregido a continuación y documenta:**

 1. El código que particiona por variables discretas adecuadas (por ejemplo, extraídas de las fechas).

 2. Un comando de terminal o script que verifique la estructura de directorios y el tamaño de los archivos generados. Recuerda que en jupyter para ejecutar un comando de consola se usa el símbolo `!` al inicio de la celda.

In [7]:
# Escribe tu código aquí para el punto 1 aquí
print("--- Caso 4: Particionamiento por variables discretas (year y month) ---")
inicio = time.time()

carpeta_salida = "parquet_adecuado"

(
    pl.scan_parquet(archivo_parquet)

    # 1. Convertir el texto a fecha-hora nativa y extraer año y mes.
    #    Son columnas DISCRETAS de baja cardinalidad: pocas combinaciones posibles
    #    (años del dataset x 12 meses), en lugar de millones de valores distintos.
    .with_columns(
        pl.col("pickup_datetime")
        .str.to_datetime("%Y-%m-%d %H:%M:%S UTC", strict=False)
        .alias("pickup_dt")
    )
    .with_columns(
        pl.col("pickup_dt").dt.year().alias("year"),
        pl.col("pickup_dt").dt.month().alias("month"),
    )

    # 2. Quitar filas con fecha inválida, para no crear una partición "null"
    .filter(pl.col("year").is_not_null())

    # 3. Escritura por lotes (sin cargar todo en RAM), particionada por clave.
    #    En Polars 2.0, PartitionByKey(by=...) se reemplaza por PartitionBy(key=...).
    #    Crea carpetas estilo Hive: parquet_adecuado/year=2015/month=6/...
    .sink_parquet(
        pl.PartitionBy(carpeta_salida, key=["year", "month"]),
        mkdir=True,
    )
)

print(f"Particionado terminado en {time.time() - inicio:.2f} s")

--- Caso 4: Particionamiento por variables discretas (year y month) ---
Particionado terminado en 103.75 s


In [8]:
# Escribe los comandos de terminal aquí
# 1. Estructura de carpetas (year y month)
!find parquet_adecuado -type d | sort
# 2. Número total de archivos Parquet generados
!find parquet_adecuado -name "*.parquet" | wc -l
# 3. Tamaño de cada archivo, de mayor a menor
!find parquet_adecuado -name "*.parquet" -exec ls -lh {} \; | sort -k5 -hr
# 4. Tamaño total de la carpeta
!du -sh parquet_adecuado

parquet_adecuado
parquet_adecuado/year=2009
parquet_adecuado/year=2009/month=1
parquet_adecuado/year=2009/month=10
parquet_adecuado/year=2009/month=11
parquet_adecuado/year=2009/month=12
parquet_adecuado/year=2009/month=2
parquet_adecuado/year=2009/month=3
parquet_adecuado/year=2009/month=4
parquet_adecuado/year=2009/month=5
parquet_adecuado/year=2009/month=6
parquet_adecuado/year=2009/month=7
parquet_adecuado/year=2009/month=8
parquet_adecuado/year=2009/month=9
parquet_adecuado/year=2010
parquet_adecuado/year=2010/month=1
parquet_adecuado/year=2010/month=10
parquet_adecuado/year=2010/month=11
parquet_adecuado/year=2010/month=12
parquet_adecuado/year=2010/month=2
parquet_adecuado/year=2010/month=3
parquet_adecuado/year=2010/month=4
parquet_adecuado/year=2010/month=5
parquet_adecuado/year=2010/month=6
parquet_adecuado/year=2010/month=7
parquet_adecuado/year=2010/month=8
parquet_adecuado/year=2010/month=9
parquet_adecuado/year=2011
parquet_adecuado/year=2011/month=1
parquet_adecuado/year

 ---

 ## Caso 5: Resiliencia de Esquemas y Valores Faltantes en Producción



 **Escenario:** Al procesar un nuevo archivo de datos, la inferencia automática de esquemas falla o produce comportamientos inconsistentes debido a la presencia de valores nulos e incoherencias de tipos en la columna `passenger_count`.



 > **Recomendación:** En entornos de producción no se debe confiar ciegamente en la inferencia automática de esquemas sobre archivos planos (CSV). Utiliza la opción `schema_overrides` dentro de `scan_csv()` para definir explícitamente el esquema de datos y aplica expresiones de imputación (`.fill_null()`) o filtrado (`.drop_nulls()`) dentro de la consulta perezosa.

In [9]:
# CÓDIGO INEFICIENTE (CASO 5)
print("--- Caso 5: Inferencia rígida de esquema sin tratamiento de nulos ---")
try:
    # Ineficiencia: Dependencia de la inferencia de tipos sin manejo de esquema ni nulos
    resultado_esquema = (
        pl.scan_csv(archivo_csv)
        .select(pl.col("passenger_count").mean())
        .collect(streaming=True)
    )
    print("Promedio de pasajeros:", resultado_esquema)
except Exception as e:
    print(f"Error detectado en la inferencia de esquema: {e}")


--- Caso 5: Inferencia rígida de esquema sin tratamiento de nulos ---
Error detectado en la inferencia de esquema: the argument 'streaming' for 'LazyFrame.collect' was deprecated in version 1.25.0 and has been removed in version 2.0. Use `engine="streaming"` instead.


 ### [Espacio de Respuesta - Caso 5]

 **Escribe tu código corregido a continuación y documenta:**

 1. Consulta como usar `schema_overrides`.
 2. Define explícitamente los tipos usando `schema_overrides`.
 3. Aplica la estrategia para la gestión de valores nulos dentro del marco perezoso.

Escribe con tus propias palabras que es y como usar `schema_overrides`

In [10]:
# Escribe tu código aquí para el punto 2
print("--- Caso 5: Esquema explícito con schema_overrides ---")

# Definir explícitamente los tipos con schema_overrides.
# Solo se fuerzan las columnas que usamos; el resto se infiere.
# passenger_count -> Int16: entero pequeño, sin riesgo de desbordar
# (Int8 llega solo a 127 y este dataset tiene valores atípicos altos).
tipos = {
    "fare_amount": pl.Float64,
    "pickup_datetime": pl.String,
    "passenger_count": pl.Int16,
}

lazy_esquema = pl.scan_csv(archivo_csv, schema_overrides=tipos)

# Verificar el esquema resultante (no lee datos, solo consulta el plan)
print(lazy_esquema.collect_schema())

--- Caso 5: Esquema explícito con schema_overrides ---
Schema({'key': String, 'fare_amount': Float64, 'pickup_datetime': String, 'pickup_longitude': Float64, 'pickup_latitude': Float64, 'dropoff_longitude': Float64, 'dropoff_latitude': Float64, 'passenger_count': Int16})


In [11]:
# Escribe tu código aquí para el punto 3
print("--- Caso 5: Gestión de nulos dentro del plan perezoso ---")

# Usa lazy_esquema, definido en el punto 2 (con schema_overrides).
resultado_esquema = (
    lazy_esquema
    .select(
        pl.len().alias("total_filas"),
        pl.col("passenger_count").null_count().alias("nulos_passenger_count"),

        # a) Estrategia de filtrado: mean() ya ignora los nulos, lo que equivale
        #    a aplicar .drop_nulls() sobre esa columna.
        pl.col("passenger_count").drop_nulls().mean().alias("media_sin_nulos"),

        # b) Estrategia de imputación: reemplazar nulos por un valor razonable
        #    (aquí 1 pasajero, el valor más frecuente en viajes de taxi).
        pl.col("passenger_count").fill_null(1).mean().alias("media_imputada"),
    )
    .collect(engine="streaming")   # Polars >= 2.0: reemplaza a streaming=True
)

print(resultado_esquema)

--- Caso 5: Gestión de nulos dentro del plan perezoso ---
shape: (1, 4)
┌─────────────┬───────────────────────┬─────────────────┬────────────────┐
│ total_filas ┆ nulos_passenger_count ┆ media_sin_nulos ┆ media_imputada │
│ ---         ┆ ---                   ┆ ---             ┆ ---            │
│ u32         ┆ u32                   ┆ f64             ┆ f64            │
╞═════════════╪═══════════════════════╪═════════════════╪════════════════╡
│ 55423856    ┆ 0                     ┆ 1.68538         ┆ 1.68538        │
└─────────────┴───────────────────────┴─────────────────┴────────────────┘


 ---

 ## Caso 6: Interoperabilidad Eficiente en Machine Learning (Apache Arrow / Zero-Copy)

 **Escenario:** Al preparar la matriz de características para entrenar un modelo predictivo en la librería ara machine learning `Scikit-Learn`, el analista invoca la función `.to_pandas()`, provocando una duplicación innecesaria del consumo de memoria RAM.

 > **Recomendación:** Polars gestiona sus datos internamente mediante el formato de memoria **Apache Arrow**. La conversión a Pandas genera una copia completa en memoria bajo bloques de NumPy. Para pasar datos a algoritmos de Machine Learning de forma eficiente, utiliza la exportación nativa a tablas Arrow (`.to_arrow()`) o arreglos NumPy directos, aprovechando la transferencia sin copia (*Zero-Copy*).

In [12]:
# CÓDIGO INEFICIENTE (CASO 6)
print("--- Caso 6: Duplicación de memoria al convertir a Pandas ---")
inicio = time.time()

# Ineficiencia: Carga a Polars y posterior conversión ansiosa a Pandas
df_polars = pl.scan_parquet(archivo_parquet).select(["fare_amount", "passenger_count"]).collect()
df_pandas = df_polars.to_pandas()  # Genera duplicación de datos en RAM

tiempo_conversion = time.time() - inicio
print(f"Matriz procesada y convertida a Pandas en {tiempo_conversion:.2f} s")


--- Caso 6: Duplicación de memoria al convertir a Pandas ---
Matriz procesada y convertida a Pandas en 21.12 s


 ### [Espacio de Respuesta - Caso 6]

 **Escribe tu código corregido a continuación y documenta:**

 1. Escribe el código para la exportación a Apache Arrow o arreglos NumPy directa.
 2. Explica conceptualmente con tus propias palabras cómo la especificación de memoria de Apache Arrow evita la duplicación de bytes en la RAM.

In [14]:
# Escribe tu código aquí
print("--- Caso 6: Exportación sin duplicar memoria (Arrow / NumPy) ---")
inicio = time.time()

# 1. Lectura perezosa de solo las 2 columnas necesarias; collect() al final.
df_polars = (
    pl.scan_parquet(archivo_parquet)
    .select(["fare_amount", "passenger_count"])
    .collect()
)

# 2. Exportación a Apache Arrow: reutiliza los buffers de Polars (Zero-Copy).
tabla_arrow = df_polars.to_arrow()

# 3. Para NumPy hace falta memoria contigua: los datos vienen en varios bloques (chunks),
#    así que se unifican con rechunk() antes de exportar.
#    Este rechunk() sí hace una copia, pero UNA sola vez y solo de 2 columnas
#    (to_pandas() copia de forma más pesada).
df_polars = df_polars.rechunk()

# 4. Exportación a NumPy columna por columna.
#    allow_copy=False comprueba que, ya con un solo bloque, la conversión es Zero-Copy.
y = df_polars["fare_amount"].to_numpy(allow_copy=False)
X = df_polars["passenger_count"].to_numpy(allow_copy=False)

tiempo_arrow = time.time() - inicio
print(f"Exportación a Arrow y NumPy en {tiempo_arrow:.2f} s")
print(f"Tabla Arrow: {tabla_arrow.num_rows} filas | NumPy: {y.shape}, {X.shape}")

# 5. Comparación con la versión ineficiente
print(f"Tiempo con to_pandas(): {tiempo_conversion:.2f} s")

--- Caso 6: Exportación sin duplicar memoria (Arrow / NumPy) ---


: 

Explica el punto 2 aquí con tus propias palabras

La idea en corto: Arrow define una forma estándar de ordenar los datos en memoria: cada columna se guarda como un bloque continuo de bytes, con su propio bloque aparte para los nulos. Como Polars, NumPy y otras librerías pueden entender ese mismo orden, no hace falta convertir los datos de un formato a otro. Basta con que la otra librería apunte a los mismos bytes que ya están en la RAM (Zero-Copy).

Por qué to_pandas() sí duplica: Pandas guarda los datos en bloques de NumPy con su propia organización. Para pasar de Arrow a ese formato, hay que reservar memoria nueva y copiar todos los valores. Los mismos datos quedan dos veces en la RAM.

Ejemplo para entenderlo: es como tener un libro y, en vez de fotocopiarlo para que otra persona lo lea, simplemente le dices en qué estante está. Las dos personas leen el mismo ejemplar.

Matiz que viste en tu práctica: el Zero-Copy solo funciona si los datos están en un bloque continuo y sin nulos. Por eso allow_copy=False falló antes de usar rechunk().

Este punto pide que lo escribas con tus propias palabras. Úsalo para entender el concepto y redáctalo tú. Si quieres, pásame tu borrador y te confirmo si está correcto.

---

Felicitaciones, haz terminado el taller, ahora entiendes los conceptos claves de cualquier ingeniero de datos que trabaja en una empresa mediana-grande.